In [1]:
import sys
import os

import pandas as pd
import json
import re

In [2]:
!python --version

Python 3.9.0


In [3]:
# настройка вывода jupyter
from IPython.display import display, HTML
import IPython

ip = IPython.get_ipython()
ip.display_formatter.formatters['text/plain'].max_seq_length = 20

## Задание 1 (синтаксис)

In [4]:
# Маппер
# MAP_LATIN_TO_CYR = str.maketrans({
#     'A': 'А', 'B': 'В', 'E': 'Е', 'K': 'К', 'M': 'М',
#     'H': 'Н', 'O': 'О', 'P': 'Р', 'C': 'С', 'T': 'Т',
#     'Y': 'У', 'X': 'Х',
# })

In [5]:
# 1 буква + 3 цифры + 2 буквы + 2–3 цифры региона
# CAR_ID_RE = r'([АВЕКМНОРСТУХ])(\d{3})([АВЕКМНОРСТУХ]{2})(\d{2,3})\Z'

In [6]:

def check_car_id(car_id):

    if not isinstance(car_id, str):
        raise "Тип данных не текст"

    MAP_LATIN_TO_CYR = str.maketrans({
        'A': 'А', 'B': 'В', 'E': 'Е', 'K': 'К', 'M': 'М',
        'H': 'Н', 'O': 'О', 'P': 'Р', 'C': 'С', 'T': 'Т',
        'Y': 'У', 'X': 'Х',
    })
    normalized = car_id.strip().upper().translate(MAP_LATIN_TO_CYR)

    CAR_ID_RE = r'([АВЕКМНОРСТУХ])(\d{3})([АВЕКМНОРСТУХ]{2})(\d{2,3})\Z'
    match = re.match(CAR_ID_RE, normalized)
    if not match:
        return 'Номер не валиден.'

    first_letter, digits, letters, region = match.groups()
    number = f'{first_letter}{digits}{letters}'
    return f'Номер {number} валиден. Регион: {region}.'


In [7]:

# ---- Проверка примеров ----
if __name__ == '__main__':
    tests = [
        'А222ВС96',
        'А123ВЕ777',
        'АБ22ВВ193',
        'A222BC96',
        'А222ВС9',
        'А222ВС9699',
        'Я222ВС96',
        '',
    ]
    for t in tests:
        print(f'{t!r:>18} -> {check_car_id(t)}')

        'А222ВС96' -> Номер А222ВС валиден. Регион: 96.
       'А123ВЕ777' -> Номер А123ВЕ валиден. Регион: 777.
       'АБ22ВВ193' -> Номер не валиден.
        'A222BC96' -> Номер А222ВС валиден. Регион: 96.
         'А222ВС9' -> Номер не валиден.
      'А222ВС9699' -> Номер не валиден.
        'Я222ВС96' -> Номер не валиден.
                '' -> Номер не валиден.


### Пограничные случаи:

1. Латиница A/B/C - по правилу нормализации

In [8]:
# нормализацию имеется в функции через мапер
check_car_id('A222BC96')

'Номер А222ВС валиден. Регион: 96.'

2. Регион из 1 цифры

In [9]:
check_car_id('А222ВС9')

'Номер не валиден.'

3. Регион из 4 цифры

In [10]:
check_car_id('А222ВС9699')

'Номер не валиден.'

4. Недопустимая буква Я

In [11]:
check_car_id('Я222ВС96')

'Номер не валиден.'

5.  Номер не валиден.

In [12]:
check_car_id('')

'Номер не валиден.'

## Задание 2

In [13]:
os.listdir()

['.ipynb_checkpoints', 'hw4.ipynb', 'logs_hw4.txt']

In [14]:
# Смотрю содержимое файла
with open('logs_hw4.txt', mode='r') as f:
    for i, v in enumerate(f.readlines()):
        if i > 20:
            break
        print(v.strip())

2024-03-15 10:51:25 WARN user=user+tag@gmail.com action=login ip=203.0.113.42 extra=foo note=bar
2024-13-01 12:00:00 INFO user=ivan@mail.ru action=login ip=1.1.1.1
2024-03-15 11:46:57 DEBUG user=petr.sidorov@yandex.ru action=login ip=8.8.8.8 msg=retry retry retry please please
2024-03-15 09:36:54 WARN user=petr.sidorov@yandex.ru action=retry ip=999.1.1.1
2024-03-15 09:07:09 ERROR user=user@domain. action=delete ip=0.0.0.0
2024-03-15 10:28:11 ERROR user=@onlydomain.com action=pay ip=300.300.300.300 card=4111-1111-1111-11111
2024-03-15 10:04:06 ERROR user=user+tag@gmail.com action=pay ip=127.0.0.1 card=4111-1111-1111-1111-1111
2024-03-15 10:59:38 DEBUG user=a.b@domain.co action=login ip=255.255.255.255 extra=foo note=bar
2024-03-15 12:16:08 INFO user=x@y.z action=login ip=203.0.113.42 path=/api/v1/users
2024-03-15 08:40:48 ERROR user=petr.sidorov@yandex.ru action=export ip=192.168.0.12
2024-03-15 08:05:06 DEBUG user=petr.sidorov@yandex.ru action=pay ip=255.255.255.255
2024-03-15 09:05:44

In [15]:
def parse_logs(text):
    if not text or not text.strip():
        return []

    result = []

    LOG_RE = re.compile(
        r'^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\s+'
        r'(?P<level>[A-Z]+)\s+'
        r'(?P<rest>.*)$'
    )
    
    EMAIL_RE = re.compile(
        r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9-]+(\.[A-Za-z0-9-]+)*\.[A-Za-z]{2,}$'
    )
    
    IPV4_RE = re.compile(
        r'^(?:(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)\.){3}'
        r'(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)$'
    )
    
    # Карта: ровно 16 цифр, разбитых дефисами 4-4-4-4
    CARD_RE = re.compile(r'^\d{4}-\d{4}-\d{4}-\d{4}$')
    
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue

        
        m = LOG_RE.match(line)
        if not m:
            continue

        ts = m.group('ts')
        level = m.group('level')
        rest = m.group('rest')

        errors = []

        fields = dict(re.findall(r'\b(user|action|ip|card)=(\S+)', rest))
        user     = fields.get('user')
        action   = fields.get('action')
        ip       = fields.get('ip')
        card_raw = fields.get('card')

        if user is None:
            errors.append('missing_user')
        elif not EMAIL_RE.match(user):
            errors.append('invalid_email')

        if action is None:
            errors.append('missing_action')

        if ip is None:
            errors.append('missing_ip')
        elif not IPV4_RE.match(ip):
            errors.append('invalid_ip')

        card = None
        if card_raw is not None:
            if CARD_RE.match(card_raw):
                card = f'****-****-****-{card_raw[-4:]}'
            else:
                errors.append('invalid_card')

        cleaned = re.sub(r'\b\w+=\S+', ' ', rest)
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()

        if cleaned:
            message = re.sub(
                r'\b(\w+)(?:\s+\1\b)+',
                r'\1',
                cleaned,
                flags=re.IGNORECASE
            )
        else:
            message = None

        result.append({
            'ts': ts,
            'level': level,
            'user': user,
            'action': action,
            'ip': ip,
            'card': card,
            'errors': errors,
            'message': message,
        })

    return result

In [16]:
data = None
with open('logs_hw4.txt', mode='r', encoding='utf-8') as f:
    data = parse_logs(f.read())

In [17]:
data[:20]

[{'ts': '2024-03-15 10:51:25',
  'level': 'WARN',
  'user': 'user+tag@gmail.com',
  'action': 'login',
  'ip': '203.0.113.42',
  'card': None,
  'errors': [],
  'message': None},
 {'ts': '2024-13-01 12:00:00',
  'level': 'INFO',
  'user': 'ivan@mail.ru',
  'action': 'login',
  'ip': '1.1.1.1',
  'card': None,
  'errors': [],
  'message': None},
 {'ts': '2024-03-15 11:46:57',
  'level': 'DEBUG',
  'user': 'petr.sidorov@yandex.ru',
  'action': 'login',
  'ip': '8.8.8.8',
  'card': None,
  'errors': [],
  'message': 'retry please'},
 {'ts': '2024-03-15 09:36:54',
  'level': 'WARN',
  'user': 'petr.sidorov@yandex.ru',
  'action': 'retry',
  'ip': '999.1.1.1',
  'card': None,
  'errors': ['invalid_ip'],
  'message': None},
 {'ts': '2024-03-15 09:07:09',
  'level': 'ERROR',
  'user': 'user@domain.',
  'action': 'delete',
  'ip': '0.0.0.0',
  'card': None,
  'errors': ['invalid_email'],
  'message': None},
 {'ts': '2024-03-15 10:28:11',
  'level': 'ERROR',
  'user': '@onlydomain.com',
  'acti

### Пограничные случаи

In [18]:
data = (
    '             \t\t       \n',
    '2024-13-01 12:00:00 INFO user= action= ip=1.1.1.1\n',
    '2024-03-15 11:46:57 DEBUG ip=8.8.8.8 msg=retry retry retry please please\n',
    '2024-03-15 09:36:54 WARN user=petr.sidorov@yandex.ru action=retry ip=999.1.1.1\n',
    '2024-03-15 09:07:09 ERROR user=user@domain. action=delete ip=0.0.0.0\n',
    '2024-03-15 10:28:11 ERROR user=@onlydomain.com action=pay ip=300.300.300.300 card=4111111111111111\n',
    'error error error timeout\n',
    
)
for i in data:
    print(parse_logs(i))

[]
[{'ts': '2024-13-01 12:00:00', 'level': 'INFO', 'user': None, 'action': None, 'ip': '1.1.1.1', 'card': None, 'errors': ['missing_user', 'missing_action'], 'message': 'user= action='}]
[{'ts': '2024-03-15 11:46:57', 'level': 'DEBUG', 'user': None, 'action': None, 'ip': '8.8.8.8', 'card': None, 'errors': ['missing_user', 'missing_action'], 'message': 'retry please'}]
[{'ts': '2024-03-15 09:36:54', 'level': 'WARN', 'user': 'petr.sidorov@yandex.ru', 'action': 'retry', 'ip': '999.1.1.1', 'card': None, 'errors': ['invalid_ip'], 'message': None}]
[{'ts': '2024-03-15 09:07:09', 'level': 'ERROR', 'user': 'user@domain.', 'action': 'delete', 'ip': '0.0.0.0', 'card': None, 'errors': ['invalid_email'], 'message': None}]
[{'ts': '2024-03-15 10:28:11', 'level': 'ERROR', 'user': '@onlydomain.com', 'action': 'pay', 'ip': '300.300.300.300', 'card': None, 'errors': ['invalid_email', 'invalid_ip', 'invalid_card'], 'message': None}]
[]
